In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, trim, lower, abs, when

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
orders_df = spark.read.parquet("/user/hive/warehouse/sclsepb4532_raw.db/orders", header=True, inferSchema=True)
payments_df = spark.read.parquet("hdfs://master:9000/user/hive/warehouse/sclsepb4532_raw.db/payments", header=True, inferSchema=True)

26/11/06 04:16:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 04:16:25 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [2]:
precision_threshold = 0.01

# Filter to keep ONLY records where the payment exactly matches the order value
strict_payments_df = payments_df.join(
    orders_df.select("order_id", "total_amount"), 
    on="order_id", 
    how="inner"
).filter(abs(col("amount_paid") - col("total_amount")) <= precision_threshold)

In [3]:
payments_df = strict_payments_df.dropDuplicates(["payment_id"]) 

In [4]:
df = payments_df.withColumn("failed_payment_flag", when(col("payment_status") == "Failed", 1).otherwise(0))

In [5]:
df.write.mode("overwrite").saveAsTable("sclsepb4532_staging.payments_stg")

26/11/06 04:18:05 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
